# Gemma 4 for Everything in KerasHub!

**Author:** [Sachin Prasad](https://github.com/sachinprasadhs)<br>
**Date created:** 2026/04/14<br>
**Last modified:** 2026/04/17<br>
**Description:** A practical Gemma 4 guide covering text, image, video, audio, object detection, OCR, coding, function calling, thinking, and reasoning in KerasHub.<br>
**Accelerator:** GPU

## Overview

Gemma 4 is a family of multimodal open models built for long-context reasoning, coding, and agentic workflows. In KerasHub, Gemma 4 presets are exposed through the standard `from_preset()` API, making it straightforward to load the model family for text, image, video, and audio-aware workflows from a single interface.

Gemma 4 introduces key capability and architectural advancements:

- Reasoning: All models in the family are designed as highly capable reasoners, with configurable thinking modes.
- Extended multimodalities: All Gemma 4 models process text and images with variable aspect ratio and resolution support, and the full family supports video-style frame understanding. Audio is available on the E2B and E4B models.
- Diverse and efficient architectures: Offers dense and Mixture-of-Experts variants across multiple sizes for scalable deployment.
- Optimized for on-device use: Smaller models are designed for efficient local execution on laptops and mobile devices.
- Increased context window: The small models feature a 128K context window, while the medium models support 256K.
- Enhanced coding and agentic capabilities: Strong coding performance is paired with native function-calling support for autonomous workflows.
- Native system prompt support: Gemma 4 supports the system role directly, enabling more structured and controllable conversations.

This guide focuses on practical usage examples in KerasHub. It covers text generation, image captioning, object detection-style JSON parsing with box overlays, audio transcription, function calling, coding, thinking and reasoning mode, multi-image comparison, video-oriented prompting, OCR, and speech translation.

## Setup

Install the latest Keras and KerasHub packages first.

```bash
pip install -U keras keras-hub
pip install -U soundfile scipy requests pillow matplotlib
```

To load Gemma 4 from Kaggle, make sure your environment exposes `KAGGLE_USERNAME` and `KAGGLE_KEY`, and that your account has accepted the Gemma 4 model license.

Keras lets you choose the backend that fits your environment. This notebook defaults to JAX, but you can switch to TensorFlow or PyTorch by setting `KERAS_BACKEND` before importing Keras.

Since this guide exercises the full range of Gemma 4 capabilities — from text and image captioning to video, audio, detection, coding, and thinking mode — many generation calls use a high `max_length` to produce meaningful outputs. For a smooth experience, running on a high-end GPU such as an H100 is strongly recommended.

In [ ]:
import os
os.environ.setdefault("KERAS_BACKEND", "jax")

import json
import re
from io import BytesIO

import keras
import keras_hub
import matplotlib.pyplot as plt
import numpy as np
import requests
import soundfile as sf
from PIL import Image

KAGGLE_PRESET = "kaggle://keras/gemma4/keras/gemma4_instruct_2b"
AUDIO_PATH = (
    "keras_hub/src/tests/test_data/audio_transcription_tests/"
    "female_short_voice_clip_17sec.wav"
)
IMAGE_URL = "http://images.cocodataset.org/val2017/000000039769.jpg"
IMAGE_URL_2 = "http://images.cocodataset.org/val2017/000000001761.jpg"
OCR_IMAGE = None

PROMPT_TEXT = (
    "<start_of_turn>user\n"
    "What is the capital of France?"
    "<end_of_turn>\n<start_of_turn>model\n"
)

PROMPT_IMAGE = (
    "<start_of_turn>user\n\n<|image|>\n"
    "Describe what you see in this image in one sentence."
    "<end_of_turn>\n<start_of_turn>model\n"
)

PROMPT_DETECTION = (
    "<start_of_turn>user\n\n<|image|>\n"
    "Identify every cat in this image. First give one short sentence, then return a JSON array inside a ```json``` block. "
    "Each item must contain `box_2d` and `label`, and `box_2d` must be in Gemma 4's 0-1000 coordinate space in [ymin, xmin, ymax, xmax] order."
    "<end_of_turn>\n<start_of_turn>model\n"
)

PROMPT_AUDIO = (
    "<|turn>user\n"
    "<|audio|>"
    "Transcribe the following speech segment in its original language. "
    "Follow these specific instructions for formatting the answer:\n"
    "* Only output the transcription, with no newlines.\n"
    "* When transcribing numbers, write the digits, i.e. write 1.7 and not one point seven, and write 3 instead of three.<turn|>\n"
    "<|turn>model\n"
)

## Helper utilities

The next cell keeps the rest of the notebook readable. In addition to loading images and audio, it includes a parser for Gemma 4 detection-style responses.

Gemma 4 may emit a short natural-language preamble before the JSON payload. The helper extracts the JSON block, rescales the 0-1000 coordinates back to pixel space, and renders boxes with `keras.visualization.draw_bounding_boxes()`.

In [ ]:
def load_image(*sources):
    """Load one or more images from URLs or local file paths."""

    def _load_one(source):
        if source.startswith(("http://", "https://")):
            response = requests.get(source, timeout=30)
            response.raise_for_status()
            return Image.open(BytesIO(response.content)).convert("RGB")
        return Image.open(source).convert("RGB")

    if not sources:
        return _load_one(IMAGE_URL)
    images = [_load_one(source) for source in sources]
    return images[0] if len(images) == 1 else images


def display_images(images, titles=None, figsize=(8, 6)):
    if not isinstance(images, (list, tuple)):
        images = [images]
    titles = titles or [None] * len(images)
    plt.figure(figsize=(figsize[0] * len(images), figsize[1]))
    for index, (image, title) in enumerate(zip(images, titles), start=1):
        plt.subplot(1, len(images), index)
        plt.imshow(image)
        plt.axis("off")
        if title:
            plt.title(title)
    plt.show()


def load_audio(path):
    raw, sample_rate = sf.read(path)
    if raw.ndim > 1:
        raw = raw.mean(axis=1)
    if sample_rate != 16000:
        from scipy import signal

        raw = signal.resample(raw, int(len(raw) * 16000 / sample_rate))
    return raw.astype(np.float32)


def strip_prompt(output, prompt):
    if isinstance(output, list):
        output = output[0]
    if output.startswith(prompt):
        return output[len(prompt):]
    for marker in ("<start_of_turn>model\n", "<|turn>model\n"):
        index = output.rfind(marker)
        if index != -1:
            return output[index + len(marker):]
    return output


def extract_json_block(text):
    fenced_match = re.search(r"```json\s*(\[.*?\])\s*```", text, flags=re.DOTALL)
    if fenced_match:
        return json.loads(fenced_match.group(1))

    bracket_match = re.search(r"(\[\s*\{.*\}\s*\])", text, flags=re.DOTALL)
    if bracket_match:
        return json.loads(bracket_match.group(1))

    raise ValueError("Could not find a JSON detection block in the model output.")


def gemma_boxes_to_keras_prediction(image, detections):
    width, height = image.size
    label_to_id = {}
    boxes = []
    labels = []

    for detection in detections:
        label = detection.get("label", "object")
        if label not in label_to_id:
            label_to_id[label] = len(label_to_id)

        y_min, x_min, y_max, x_max = detection["box_2d"]
        boxes.append([
            y_min / 1000.0 * height,
            x_min / 1000.0 * width,
            y_max / 1000.0 * height,
            x_max / 1000.0 * width,
        ])
        labels.append(label_to_id[label])

    prediction = {
        "boxes": np.array([boxes], dtype="float32"),
        "labels": np.array([labels], dtype="int32"),
    }
    class_mapping = {value: key for key, value in label_to_id.items()}
    return prediction, class_mapping


def render_detection_result(image, raw_output):
    detections = extract_json_block(raw_output)
    prediction, class_mapping = gemma_boxes_to_keras_prediction(image, detections)
    image_batch = np.expand_dims(np.array(image), axis=0)

    rendered = keras.visualization.draw_bounding_boxes(
        image_batch,
        prediction,
        bounding_box_format="yxyx",
        class_mapping=class_mapping,
        color=(255, 235, 59),
        line_thickness=3,
        text_thickness=2,
        font_scale=0.8,
    )

    plt.figure(figsize=(8, 8))
    plt.imshow(rendered[0].astype("uint8"))
    plt.axis("off")
    plt.show()
    return detections

## Load the Gemma 4 preset

This guide uses `gemma4_instruct_2b` from Kaggle. It is a practical starting point because it supports text, image, video-style prompting, and audio-enabled workflows in a relatively small checkpoint.

In [ ]:
print(f"Loading preset: {KAGGLE_PRESET}")
model = keras_hub.models.Gemma4CausalLM.from_preset(
    KAGGLE_PRESET,
    dtype="bfloat16",
)

print("audio_converter :", model.preprocessor.audio_converter)
print("audio_encoder   :", model.backbone.audio_encoder)

## 1. Text generation

Start with a simple usage example. This confirms that the preset, tokenizer, and prompt formatting are all working before moving into multimodal prompts.

In [ ]:
text_output = model.generate({"prompts": [PROMPT_TEXT]}, max_length=128)
print(strip_prompt(text_output, PROMPT_TEXT))

## 2. Image captioning

Gemma 4 supports image understanding directly through `Gemma4CausalLM.generate()`. A simple captioning prompt is a good way to verify multimodal input handling before moving on to more structured visual tasks.

In [ ]:
image = load_image()
display_images(image, titles=["Input image"])

image_output = model.generate(
    {"prompts": [PROMPT_IMAGE], "images": [image]},
    max_length=2048,
)
print(strip_prompt(image_output, PROMPT_IMAGE))

## 3. Object detection-style localization

Gemma 4 can localize objects by returning structured JSON alongside natural-language output. In practice, the model often produces a short explanation before the JSON payload, so it is useful to parse the fenced block explicitly.

In this section, the model identifies cats in the image, the JSON is extracted, the normalized 0-1000 coordinates are rescaled to image pixels, and the boxes are rendered with Keras visualization utilities.

In [ ]:
detection_output = model.generate(
    {"prompts": [PROMPT_DETECTION], "images": [image]},
    max_length=1024,
)
raw_detection_text = strip_prompt(detection_output, PROMPT_DETECTION)
print(raw_detection_text)

parsed_detections = render_detection_result(image, raw_detection_text)
parsed_detections

## 4. Audio transcription

Audio input is available on the E2B and E4B Gemma 4 checkpoints. Video-style understanding is supported across the model family, but audio is limited to these smaller variants. The prompt below makes the task explicit and specifies the exact output formatting.

In [ ]:
if not os.path.exists(AUDIO_PATH):
    print(f"Audio file not found at {AUDIO_PATH}.")
else:
    audio = load_audio(AUDIO_PATH)
    print("Audio shape:", audio.shape, "dtype:", audio.dtype)
    audio_output = model.generate(
        {"prompts": [PROMPT_AUDIO], "audio": [audio]},
        max_length=512,
    )
    print(strip_prompt(audio_output, PROMPT_AUDIO))

## 5. Function calling

Gemma 4 supports native function calling for tool-augmented workflows. The basic pattern is consistent across applications: declare the tool, let the model emit a tool call, execute the tool externally, append the tool response, and then continue generation.

In [ ]:
PROMPT_FUNC_CALL = (
    "<|turn>system\n"
    "You are a helpful assistant."
    "<|tool>declaration:get_current_weather{"
    "description:<|\"|>Returns the current weather for a given city.,"
    "parameters:{"
    "location:{type:<|\"|>string<|\"|>,description:<|\"|>The city name<|\"|>}"
    "}}"
    "<tool|><turn|>\n"
    "<|turn>user\n"
    "What is the weather like in Paris right now?<turn|>\n"
    "<|turn>model\n"
)

tool_call_output = model.generate({"prompts": [PROMPT_FUNC_CALL]}, max_length=256)
tool_call_text = tool_call_output[0] if isinstance(tool_call_output, list) else tool_call_output
tool_call_text = tool_call_text.split("<|turn>model\n")[-1]
print(tool_call_text)

PROMPT_WITH_RESPONSE = (
    PROMPT_FUNC_CALL
    + "<|tool_call>call:get_current_weather{location:<|\"|>Paris<|\"|>}<tool_call|>"
    + "<|tool_response>response:get_current_weather{temperature:18,weather:<|\"|>partly cloudy<|\"|>}<tool_response|>\n"
    + "<|turn>model\n"
)

final_weather_output = model.generate({"prompts": [PROMPT_WITH_RESPONSE]}, max_length=256)
final_weather_text = final_weather_output[0] if isinstance(final_weather_output, list) else final_weather_output
final_weather_text = final_weather_text.split("<|turn>model\n")[-1]
print(final_weather_text)

## 6. Coding

Gemma 4 is also a strong coding model. You can use the same causal language model interface for code generation, refactoring, explanation, and small utility synthesis tasks.

In [ ]:
PROMPT_CODE = (
    "<start_of_turn>user\n"
    "Write a Python function named `is_palindrome` that ignores punctuation, whitespace, and letter casing. Also include a few short test calls."
    "<end_of_turn>\n<start_of_turn>model\n"
)

code_output = model.generate({"prompts": [PROMPT_CODE]}, max_length=512)
print(strip_prompt(code_output, PROMPT_CODE))

## 7. Thinking mode

Thinking mode is controlled from the system prompt. Gemma 4 can emit a thought channel before a tool call or before the final answer. In production, keep that hidden reasoning out of the visible conversation history.

The next two cells show both patterns: first, a tool-style research turn with a mocked tool response, and second, a reasoning-only math example.

In [ ]:
PROMPT_AGENT = (
    "<|turn>system\n"
    "<|think|>You are a helpful research assistant."
    "<|tool>declaration:search_web{"
    "description:<|\"|>Search the web for current information.<|\"|>,"
    "parameters:{"
    "queries:{type:<|\"|>array<|\"|>,description:<|\"|>List of search queries<|\"|>}"
    "}}"
    "<tool|><turn|>\n"
    "<|turn>user\n"
    "Research and summarize the latest news on fusion energy.<turn|>\n"
    "<|turn>model\n"
)

agent_step_1 = model.generate({"prompts": [PROMPT_AGENT]}, max_length=512)
agent_text_1 = agent_step_1[0] if isinstance(agent_step_1, list) else agent_step_1
agent_text_1 = agent_text_1.split("<|turn>model\n")[-1]
print(agent_text_1)

FAKE_TOOL_RESPONSE = (
    "<|tool_response>response:search_web{results:["
    "{title:<|\"|>ITER plasma milestone reached<|\"|>,snippet:<|\"|>ITER achieved a record plasma duration in early 2026, marking a key milestone toward sustained fusion reactions.<|\"|>},"
    "{title:<|\"|>Commonwealth Fusion Systems closes $2B round<|\"|>,snippet:<|\"|>CFS plans to demonstrate net energy gain with SPARC by 2027 and deploy commercial plants by 2030.<|\"|>},"
    "{title:<|\"|>NIF reports repeated ignition shots<|\"|>,snippet:<|\"|>The National Ignition Facility has repeated its ignition result multiple times, strengthening confidence in laser-driven fusion.<|\"|>}"
    "]}<tool_response|>\n"
    "<|turn>model\n"
)

context_with_result = PROMPT_AGENT + agent_text_1.split("<|tool_response>")[0] + FAKE_TOOL_RESPONSE
agent_step_2 = model.generate({"prompts": [context_with_result]}, max_length=512)
agent_text_2 = agent_step_2[0] if isinstance(agent_step_2, list) else agent_step_2
agent_text_2 = agent_text_2.split("<|turn>model\n")[-1]
print(agent_text_2)

In [ ]:
PROMPT_THINKING = (
    "<|turn>system\n"
    "<|think|>You are a helpful math tutor.<turn|>\n"
    "<|turn>user\n"
    "A train travels 120 km in 1.5 hours. How long will it take to travel 200 km at the same speed?<turn|>\n"
    "<|turn>model\n"
)

thinking_output = model.generate({"prompts": [PROMPT_THINKING]}, max_length=512)
thinking_text = thinking_output[0] if isinstance(thinking_output, list) else thinking_output
thinking_text = thinking_text.split("<|turn>model\n")[-1]
print(thinking_text)

## 8. Multi-image comparison

Gemma 4 can compare multiple images within a single turn. Each `<|image|>` token corresponds to one image in the `images` list, which makes it easy to build comparison, retrieval, and frame-based prompts.

In [ ]:
image_1, image_2 = load_image(IMAGE_URL, IMAGE_URL_2)
display_images([image_1, image_2], titles=["Image 1", "Image 2"])

PROMPT_COMPARE = (
    "<start_of_turn>user\n\n<|image|>\n<|image|>\n"
    "What are the key differences between these two images? Describe each image briefly, then list the differences."
    "<end_of_turn>\n<start_of_turn>model\n"
)

comparison_output = model.generate(
    {"prompts": [PROMPT_COMPARE], "images": [image_1, image_2]},
    max_length=512,
)
print(strip_prompt(comparison_output, PROMPT_COMPARE))

## 9. Video prompting

Gemma 4 supports video understanding by processing sampled frames in temporal order. In KerasHub, a practical pattern is to decode a video into representative frames, pass them as ordered images, and ask for a temporal summary, event description, or change analysis.

In [ ]:
video_frames = [image_1, image, image_2]
display_images(video_frames, titles=["Frame 1", "Frame 2", "Frame 3"], figsize=(5, 4))

PROMPT_VIDEO = (
    "<start_of_turn>user\n\n<|image|>\n<|image|>\n<|image|>\n"
    "Treat these images as consecutive video frames. Summarize what stays consistent across frames and what changes over time."
    "<end_of_turn>\n<start_of_turn>model\n"
)

video_output = model.generate(
    {"prompts": [PROMPT_VIDEO], "images": video_frames},
    max_length=512,
)
print(strip_prompt(video_output, PROMPT_VIDEO))

## 10. OCR and document parsing

Gemma 4 is strong on OCR and document understanding. For document-heavy tasks, the prompting pattern stays simple: pass the image, ask for exact extraction, and make the expected output format explicit.

In [ ]:
if OCR_IMAGE is None:
    print("Set OCR_IMAGE to a local path or URL to run this section.")
else:
    ocr_image = load_image(OCR_IMAGE)
    display_images(ocr_image, titles=["OCR input"])

    PROMPT_OCR = (
        "<start_of_turn>user\n\n<|image|>\n"
        "Extract all text from this document exactly as it appears, preserving the original layout as closely as possible."
        "<end_of_turn>\n<start_of_turn>model\n"
    )

    ocr_output = model.generate(
        {"prompts": [PROMPT_OCR], "images": [ocr_image]},
        max_length=2048,
    )
    print(strip_prompt(ocr_output, PROMPT_OCR))

## 11. Speech translation

For audio-capable Gemma 4 checkpoints, speech translation follows the same pattern as transcription: provide the audio input, specify the source and target behavior clearly, and define the output format up front.

In [ ]:
if not os.path.exists(AUDIO_PATH):
    print(f"Audio file not found at {AUDIO_PATH}.")
else:
    audio_translate = load_audio(AUDIO_PATH)
    PROMPT_TRANSLATE = (
        "<|turn>user\n"
        "<|audio|>"
        "Transcribe the following speech segment in its original language, then translate it into English.\n"
        "When formatting the answer, first output the transcription, then one newline, then output 'English: ', then the translation.<turn|>\n"
        "<|turn>model\n"
    )

    translation_output = model.generate(
        {"prompts": [PROMPT_TRANSLATE], "audio": [audio_translate]},
        max_length=512,
    )
    print(strip_prompt(translation_output, PROMPT_TRANSLATE))

## Closing notes

This guide stays close to the low-level `Gemma4CausalLM.generate()` API on purpose. It makes the prompt format explicit, shows how to pass images and audio, and demonstrates how to post-process structured output for downstream visualization.

For production usage, the main things to harden are authentication, response validation, retry handling, and guardrails around tool execution and OCR or detection post-processing.